# Credit Card Fraud Detection — Data Validation & Verification

**Dataset:** [Kaggle — Credit Card Fraud Detection (ULB MLG)](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud/data)  
**Research Reference:** [ResearchGate — Credit card fraud data analysis using proposed sampling algorithm and deep ensemble learning](https://www.researchgate.net/publication/400327841_Credit_card_fraud_data_analysis_using_proposed_sampling_algorithm_and_deep_ensemble_learning)

## Context

This notebook validates and verifies the `creditcard.csv` dataset before any modeling work.

The dataset contains credit card transactions made by European cardholders in **September 2013** over **two consecutive days**.  
All features V1–V28 are the result of **PCA transformation** to protect cardholder confidentiality.  
Only `Time`, `Amount`, and `Class` are non-transformed.

The research paper uses this exact dataset and proposes:
1. A **custom sampling algorithm** to handle severe class imbalance (only ~0.17% fraud)
2. A **deep ensemble learning** model trained on the resampled data

This notebook confirms the CSV matches all documented dataset properties before those techniques are applied.

In [1]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

CSV_PATH = '../creditcard_clean.csv'
df = pd.read_csv(CSV_PATH)

print(f'Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')

Loaded: 284,807 rows x 31 columns


## Column Descriptions

The dataset has 31 columns. Each column's role is described below.

| Column | Full Name | Description |
|--------|-----------|-------------|
| `Time` | Transaction Time | Seconds elapsed between this transaction and the first transaction in the dataset |
| `V1` | PCA Component 1 | Anonymous PCA-transformed feature (original feature confidential) |
| `V2` | PCA Component 2 | Anonymous PCA-transformed feature |
| `V3` | PCA Component 3 | Anonymous PCA-transformed feature |
| `V4` | PCA Component 4 | Anonymous PCA-transformed feature |
| `V5` | PCA Component 5 | Anonymous PCA-transformed feature |
| `V6` | PCA Component 6 | Anonymous PCA-transformed feature |
| `V7` | PCA Component 7 | Anonymous PCA-transformed feature |
| `V8` | PCA Component 8 | Anonymous PCA-transformed feature |
| `V9` | PCA Component 9 | Anonymous PCA-transformed feature |
| `V10` | PCA Component 10 | Anonymous PCA-transformed feature |
| `V11` | PCA Component 11 | Anonymous PCA-transformed feature |
| `V12` | PCA Component 12 | Anonymous PCA-transformed feature |
| `V13` | PCA Component 13 | Anonymous PCA-transformed feature |
| `V14` | PCA Component 14 | Anonymous PCA-transformed feature |
| `V15` | PCA Component 15 | Anonymous PCA-transformed feature |
| `V16` | PCA Component 16 | Anonymous PCA-transformed feature |
| `V17` | PCA Component 17 | Anonymous PCA-transformed feature |
| `V18` | PCA Component 18 | Anonymous PCA-transformed feature |
| `V19` | PCA Component 19 | Anonymous PCA-transformed feature |
| `V20` | PCA Component 20 | Anonymous PCA-transformed feature |
| `V21` | PCA Component 21 | Anonymous PCA-transformed feature |
| `V22` | PCA Component 22 | Anonymous PCA-transformed feature |
| `V23` | PCA Component 23 | Anonymous PCA-transformed feature |
| `V24` | PCA Component 24 | Anonymous PCA-transformed feature |
| `V25` | PCA Component 25 | Anonymous PCA-transformed feature |
| `V26` | PCA Component 26 | Anonymous PCA-transformed feature |
| `V27` | PCA Component 27 | Anonymous PCA-transformed feature |
| `V28` | PCA Component 28 | Anonymous PCA-transformed feature |
| `Amount` | Transaction Amount | Transaction value in EUR |
| `Class` | Fraud Label | Target variable: `1` = Fraud, `0` = Legitimate |

In [2]:
# Print column names with their positional index for reference
col_desc = {
    'Time':   'Seconds elapsed since first transaction',
    'Amount': 'Transaction value in EUR',
    'Class':  'Fraud label: 1=Fraud, 0=Legitimate'
}
for v in range(1, 29):
    col_desc[f'V{v}'] = f'PCA Component {v} (anonymous)'

desc_df = pd.DataFrame([
    {'#': i+1, 'Column': col, 'Description': col_desc.get(col, ''), 'Dtype': str(df[col].dtype)}
    for i, col in enumerate(df.columns)
])

print(desc_df.to_string(index=False))

 # Column                             Description   Dtype
 1   Time Seconds elapsed since first transaction float64
 2     V1             PCA Component 1 (anonymous) float64
 3     V2             PCA Component 2 (anonymous) float64
 4     V3             PCA Component 3 (anonymous) float64
 5     V4             PCA Component 4 (anonymous) float64
 6     V5             PCA Component 5 (anonymous) float64
 7     V6             PCA Component 6 (anonymous) float64
 8     V7             PCA Component 7 (anonymous) float64
 9     V8             PCA Component 8 (anonymous) float64
10     V9             PCA Component 9 (anonymous) float64
11    V10            PCA Component 10 (anonymous) float64
12    V11            PCA Component 11 (anonymous) float64
13    V12            PCA Component 12 (anonymous) float64
14    V13            PCA Component 13 (anonymous) float64
15    V14            PCA Component 14 (anonymous) float64
16    V15            PCA Component 15 (anonymous) float64
17    V16     

## Validation Checks

Each check below verifies the dataset matches the documented properties from Kaggle and the research paper:

| # | Check | Expected | Reason |
|---|-------|----------|--------|
| 1 | Total rows | 284,807 | Kaggle spec |
| 2 | Total columns | 31 | Time + V1–V28 + Amount + Class |
| 3 | Missing values | 0 | Complete dataset |
| 4 | Fraud count | 492 | Kaggle spec |
| 5 | Legitimate count | 284,315 | Kaggle spec |
| 6 | Fraud ratio | ~0.172% | Severe imbalance per paper |
| 7 | Time minimum | 0 | First transaction = reference |
| 8 | Time maximum | ≤ 172,800s | Two days = 48h = 172,800s |
| 9 | Negative Amount | 0 | Transactions can't be negative |
| 10 | Class values | Only 0 or 1 | Binary classification target |
| 11 | V1–V28 mean | ~0.0 | PCA produces zero-centered components |
| 12 | Duplicate rows | 0 (ideal) | Duplicates skew model training |

In [3]:
vcols = [c for c in df.columns if c.startswith('V')]
pca_max_mean = round(df[vcols].mean().abs().max(), 6)
fraud_count = int((df['Class'] == 1).sum())
legit_count = int((df['Class'] == 0).sum())
fraud_ratio = round(df['Class'].mean() * 100, 4)
dups = int(df.duplicated().sum())

checks = [
    ('Total rows',            '284807',     str(df.shape[0]),                    df.shape[0] == 284807),
    ('Total columns',         '31',          str(df.shape[1]),                    df.shape[1] == 31),
    ('Missing values',        '0',           str(int(df.isnull().sum().sum())),   df.isnull().sum().sum() == 0),
    ('Fraud count',           '492',         str(fraud_count),                    fraud_count == 492),
    ('Legitimate count',      '284315',      str(legit_count),                    legit_count == 284315),
    ('Fraud ratio (%)',       '~0.172',      str(fraud_ratio),                    0.17 < fraud_ratio < 0.18),
    ('Time min',              '0',           str(df['Time'].min()),               df['Time'].min() == 0),
    ('Time max (s)',          '<=172800',    str(df['Time'].max()),               df['Time'].max() <= 172800),
    ('Negative Amount rows',  '0',           str(int((df['Amount'] < 0).sum())),  (df['Amount'] < 0).sum() == 0),
    ('Class values only 0/1', '0 invalid',   str(int((~df['Class'].isin([0,1])).sum())), (~df['Class'].isin([0,1])).sum() == 0),
    ('V1-V28 mean ~0 (PCA)',  '<0.001',      str(pca_max_mean),                   pca_max_mean < 0.001),
    ('Duplicate rows',        '0 (ideal)',   str(dups),                           dups == 0),
]

results = pd.DataFrame(checks, columns=['Check', 'Expected', 'Value', 'Pass'])
results['Status'] = results['Pass'].map(lambda x: 'PASS' if x else 'WARN/FAIL')
results = results.drop(columns='Pass')

print(results.to_string(index=False))

                Check  Expected    Value    Status
           Total rows    284807   284807      PASS
        Total columns        31       31      PASS
       Missing values         0        0      PASS
          Fraud count       492      492      PASS
     Legitimate count    284315   284315      PASS
      Fraud ratio (%)    ~0.172   0.1727      PASS
             Time min         0      0.0      PASS
         Time max (s)  <=172800 172792.0      PASS
 Negative Amount rows         0        0      PASS
Class values only 0/1 0 invalid        0      PASS
 V1-V28 mean ~0 (PCA)    <0.001      0.0      PASS
       Duplicate rows 0 (ideal)     1081 WARN/FAIL


## Duplicate Row Removal

The dataset contains **1,081 duplicate rows** — rows where every column value is identical to another row.  
Duplicates can cause:
- **Biased training**: model sees the same sample multiple times, inflating its weight
- **Data leakage**: if a duplicate spans train/test split, test performance is artificially high
- **Skewed metrics**: precision/recall numbers become unreliable

We drop all duplicates, keeping the first occurrence.

In [4]:
before = len(df)
df_clean = df.drop_duplicates().reset_index(drop=True)
after = len(df_clean)

print(f'Before : {before:,} rows')
print(f'Removed: {before - after:,} duplicate rows')
print(f'After  : {after:,} rows')
print(f'Fraud in cleaned data  : {int((df_clean["Class"]==1).sum())} ({round(df_clean["Class"].mean()*100,4)}%)')
print(f'Legit  in cleaned data : {int((df_clean["Class"]==0).sum())}')

Before : 284,807 rows
Removed: 1,081 duplicate rows
After  : 283,726 rows
Fraud in cleaned data  : 473 (0.1667%)
Legit  in cleaned data : 283253


## Post-Clean Verification

Re-run key checks on the cleaned dataframe to confirm no data was lost unintentionally and fraud ratio is preserved.

In [5]:
post_checks = [
    ('Rows after dedup',      str(after),                              True),
    ('Missing values',        str(int(df_clean.isnull().sum().sum())), df_clean.isnull().sum().sum() == 0),
    ('Duplicate rows remain', str(int(df_clean.duplicated().sum())),   df_clean.duplicated().sum() == 0),
    ('Fraud count preserved', str(int((df_clean['Class']==1).sum())),  True),
    ('Fraud ratio (%)',       str(round(df_clean['Class'].mean()*100, 4)), True),
]

post_df = pd.DataFrame(post_checks, columns=['Check', 'Value', 'Pass'])
post_df['Status'] = post_df['Pass'].map(lambda x: 'PASS' if x else 'FAIL')
post_df = post_df.drop(columns='Pass')

print(post_df.to_string(index=False))

                Check  Value Status
     Rows after dedup 283726   PASS
       Missing values      0   PASS
Duplicate rows remain      0   PASS
Fraud count preserved    473   PASS
      Fraud ratio (%) 0.1667   PASS


## Save Outputs

Two files saved:
1. `creditcard_clean.csv` — deduplicated dataset, ready for modeling
2. `creditcard_validation_report.csv` — full validation report

Both use `;` delimiter and UTF-8 BOM encoding for native Excel compatibility.

In [6]:
import os

out_dir = os.path.dirname(os.path.abspath(CSV_PATH))

clean_path  = os.path.join(out_dir, 'creditcard_clean.csv')
report_path = os.path.join(out_dir, 'creditcard_validation_report.csv')

df_clean.to_csv(clean_path, sep=';', encoding='utf-8-sig', index=False)
results.to_csv(report_path, sep=';', encoding='utf-8-sig', index=False)

print(f'Cleaned dataset : {clean_path}')
print(f'Validation report: {report_path}')
print(f'\nDone. {after:,} rows ready for modeling.')

Cleaned dataset : D:\Kuliah\Smt 5\Data Science\P2\creditcard_clean.csv
Validation report: D:\Kuliah\Smt 5\Data Science\P2\creditcard_validation_report.csv

Done. 283,726 rows ready for modeling.
